# Detección de fallas en agentes: debugging y análisis de errores

Este notebook usa trazas y métricas para detectar problemas reales en agentes LLM y proponer correcciones.

## Objetivos
- Reconocer fallas comunes en agentes: hallucination, tool misuse, loops y respuestas incompletas.
- Analizar ejecuciones paso a paso.
- Proponer mejoras con un agente corregido.

In [1]:
!pip install pandas langchain langchain-openai wikipedia python-dotenv


[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: C:\Users\ingri\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [2]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

import wikipedia
from langchain_openai import ChatOpenAI
 
# Configurar el idioma de Wikipedia
wikipedia.set_lang("es")

# Configuración del LLM usando las vars de entorno (si están disponibles)
try:
    llm = ChatOpenAI(
        model="openai/gpt-oss-120b",
        openai_api_base=os.environ.get("GROQ_BASE_URL"),
        openai_api_key=os.environ.get("GROQ_API_KEY"),
        temperature=0
    )
    print("✅ LLM de LangChain configurado (usando credenciales de .env si existe).")
except Exception as e:
    print(f"❌ Error configurando el LLM: {e}")
    llm = None

✅ LLM de LangChain configurado (usando credenciales de .env si existe).


In [3]:
from langchain_classic.agents import tool, create_openai_tools_agent, AgentExecutor
from langchain_classic import hub
from langsmith import Client

@tool
def get_wikipedia_summary(query: str) -> str:
    """Busca en Wikipedia un tema y devuelve un resumen de 2 frases. Útil para obtener información sobre personas, lugares o conceptos."""
    try:
        return wikipedia.summary(query, sentences=2)
    except Exception as e:
        return f"Ocurrió un error: {e}"

tools = [get_wikipedia_summary]

client = Client()
prompt = client.pull_prompt("hormold/openai-functions-agent",
                            dangerously_pull_public_prompt=True)

agent = create_openai_tools_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

print("✅ Agente y herramientas listos.")

✅ Agente y herramientas listos.


## Base de conocimiento local

Aquí definimos una base de conocimiento pequeña dentro del notebook, con tres temas: agentes, observabilidad e historia.

¿Por qué usar datos locales? Porque podemos saber exactamente qué respuesta es correcta. Si el agente da otra cosa, sabemos que hay una falla, no un problema de la web o de la API.

Lo que vamos a ver con esta base:
- Cuándo el agente usa la herramienta y cuándo no.
- Qué pasa cuando la herramienta no encuentra datos.
- Si el agente inventa una respuesta en vez de decir que no sabe.

In [4]:
import random
import time
from pprint import pprint

try:
    import pandas as pd
except ImportError:
    raise ImportError('Instala pandas con `pip install pandas` antes de ejecutar este notebook.')

# Base de conocimiento local para la tarea
knowledge = {
    'agentes': 'Un agente LLM usa herramientas y memoria para completar tareas.',
    'observabilidad': 'La observabilidad captura logs, trazas y métricas.',
    'historia': 'La independencia se celebró en 1810 en muchos países de América Latina.',
}

In [ ]:
#define una clase llamada DebugAgent que simula un agente LLM con trazabilidad de decisiones y herramientas.
class DebugAgent:
    def __init__(self, knowledge): # Inicializa el agente con una base de conocimiento local.
        self.knowledge = knowledge
    # metodo que recibe una pregunta y devuelve una traza de decisiones y resultados.
    def run(self, question):
        trace = [] #lista vacia donde se guardan los pasos
        # Paso 1: decidir si usa la herramienta local (pasa como True o false) dependiendo si contiene las palabras claves
        uses_tool = 'buscar' in question.lower() or 'conocimiento' in question.lower()
        trace.append({
            'step': 1,
            'tipo': 'decision',
            'input': question,
            'output': 'usar herramienta local' if uses_tool else 'respuesta directa',
            'latencia': '0.05s',
            'tokens': 10,
            'estado': 'ok'
        })

        # Paso 2: si decide usar la herramienta, busca en la base de conocimiento local; si no encuentra, indica que usará conocimiento general.
        if uses_tool:
            found = [value for key, value in self.knowledge.items() if key in question.lower()]
            if found:
                output = found[0]
                trace.append({
                    'step': 2,
                    'tipo': 'tool',
                    'input': question,
                    'output': output,
                    'latencia': '0.10s',
                    'tokens': 0,
                    'estado': 'ok'
                })
            else:
                output = 'No encontré información local, uso conocimiento general.'
                trace.append({
                    'step': 2,
                    'tipo': 'tool',
                    'input': question,
                    'output': 'herramienta fallida',
                    'latencia': '0.10s',
                    'tokens': 0,
                    'estado': 'warning' # estado de warning porque no encontró información local
                })
                trace.append({
                    'step': 3,
                    'tipo': 'llm',
                    'input': question,
                    'output': 'Respuesta general del modelo.',
                    'latencia': '0.15s',
                    'tokens': 40,
                    'estado': 'ok'
                })
                return trace

        # Paso 3: respuesta final del LLM.
        # Ojo: esto también se ejecuta cuando NO se usó la herramienta (respuesta directa),
        # pero el mensaje de abajo dice "encontró datos locales" igual.

        output = 'El agente encontró datos locales y respondió con ellos.'
        trace.append({
            'step': 3,
            'tipo': 'llm',
            'input': question,
            'output': output,
            'latencia': '0.12s',
            'tokens': 35,
            'estado': 'ok'
        })
        return trace

In [ ]:
# Función para analizar la traza con tres reglas.
def analyze_trace(trace):
    #guarda los problemas en una lista
    issues = []
    for step in trace:

        # el agente reportó un problema al usar la herramienta
        if step['tipo'] == 'tool' and step['estado'] != 'ok':

        # si la respuesta del LLM indica que usó conocimiento general, podría ser un hallucination
            issues.append('Herramienta reportó un problema: ' + step['output'])
        if step['tipo'] == 'llm' and 'general' in step['output'].lower():
            issues.append('Posible hallucination o uso de conocimiento general.')

        # el agente respondió directo sin usar herramienta cuando debería haberla usado, se asume como error
        if step['step'] > 1 and step['tipo'] == 'llm' and trace[0]['output'] == 'respuesta directa':
            issues.append('El agente respondió directo sin usar herramienta cuando debería haberla usado.')
    return issues

# Ejecutar el agente con preguntas de prueba y analizar la traza.
agent = DebugAgent(knowledge)
questions = [
    'Busca en la base de conocimiento local qué es observabilidad.',
    '¿Qué es un agente y cómo usa herramientas?',
    'Consulta datos sobre historia.'
]

for q in questions:
    print(f'\n=== Análisis para pregunta: {q}')
    trace = agent.run(q)
    df = pd.DataFrame(trace)
    display(df)
    issues = analyze_trace(trace)
    print('Issues detectados:', issues)


=== Análisis para pregunta: Busca en la base de conocimiento local qué es observabilidad.


,step,tipo,input,output,latencia,tokens,estado
0,1,decision,Busca en la base de conocimiento local qué es ...,usar herramienta local,0.05s,10,ok
1,2,tool,Busca en la base de conocimiento local qué es ...,"La observabilidad captura logs, trazas y métri...",0.10s,0,ok
2,3,llm,Busca en la base de conocimiento local qué es ...,El agente encontró datos locales y respondió c...,0.12s,35,ok


Issues detectados: []

=== Análisis para pregunta: ¿Qué es un agente y cómo usa herramientas?


,step,tipo,input,output,latencia,tokens,estado
0,1,decision,¿Qué es un agente y cómo usa herramientas?,respuesta directa,0.05s,10,ok
1,3,llm,¿Qué es un agente y cómo usa herramientas?,El agente encontró datos locales y respondió c...,0.12s,35,ok


Issues detectados: ['El agente respondió directo sin usar herramienta cuando debería haberla usado.']

=== Análisis para pregunta: Consulta datos sobre historia.


,step,tipo,input,output,latencia,tokens,estado
0,1,decision,Consulta datos sobre historia.,respuesta directa,0.05s,10,ok
1,3,llm,Consulta datos sobre historia.,El agente encontró datos locales y respondió c...,0.12s,35,ok


Issues detectados: ['El agente respondió directo sin usar herramienta cuando debería haberla usado.']


## Propuesta de corrección

En este caso, podemos mejorar el agente al:
- reforzar la decisión de usar herramienta cuando hay palabras clave relevantes.
- validar si la herramienta encontró resultados antes de pasar al modelo.
- registrar explícitamente cuando se produce una respuesta general.

In [ ]:
# crea una clase heredada de DebugAgent llamada FixedDebugAgent que corrige los problemas detectados en la traza.
class FixedDebugAgent(DebugAgent):
    def run(self, question):
        trace = []

        # Paso 1: con any revisa todas las palbras de la lista, deveulve true si al menos una aparece en la pregunnta
        uses_tool = any(word in question.lower() for word in ['buscar', 'conocimiento', 'consulta', 'datos'])
        trace.append({
            'step': 1,
            'tipo': 'decision',
            'input': question,
            'output': 'usar herramienta local' if uses_tool else 'respuesta directa',
            'latencia': '0.05s',
            'tokens': 10,
            'estado': 'ok'
        })

        # Paso 2: si decide usar la herramienta, busca en la base de conocimiento local; si no encuentra, indica que usará conocimiento general.
        if uses_tool:
            found = [value for key, value in self.knowledge.items() if key in question.lower()]
            if found:
                output = found[0]
                trace.append({
                    'step': 2,
                    'tipo': 'tool',
                    'input': question,
                    'output': output,
                    'latencia': '0.10s',
                    'tokens': 0,
                    'estado': 'ok'
                })

                # Paso 3: respuesta final del LLM, indicando que usó la información local y no inventó información.
                trace.append({
                    'step': 3,
                    'tipo': 'llm',
                    'input': output,
                    'output': f'Usé la información local y respondí: {output}', 
                    'latencia': '0.12s',
                    'tokens': 25,
                    'estado': 'ok'
                })
                return trace
            else:
                trace.append({
                    'step': 2,
                    'tipo': 'tool',
                    'input': question,
                    'output': 'no hay datos locales',
                    'latencia': '0.10s',
                    'tokens': 0,
                    'estado': 'warning'
                })
                trace.append({
                    'step': 3,
                    'tipo': 'llm',
                    'input': 'No se encontró información local. Responde con cautela.',
                    'output': 'Informé que no encontré datos locales y no inventé información.',
                    'latencia': '0.15s',
                    'tokens': 45,
                    'estado': 'ok'
                })
                return trace

        trace.append({
            'step': 2,
            'tipo': 'llm',
            'input': question,
            'output': 'Respuesta directa sin necesidad de herramienta.',
            'latencia': '0.12s',
            'tokens': 35,
            'estado': 'ok'
        })
        return trace

# Ejecutar comparación antes/después
fixed_agent = FixedDebugAgent(knowledge)

for q in questions:
    print(f'\n=== Comparación para: {q}')
    trace_before = pd.DataFrame(DebugAgent(knowledge).run(q))
    trace_after = pd.DataFrame(fixed_agent.run(q))
    print('--- Antes ---')
    display(trace_before)
    print('--- Después ---')
    display(trace_after)


=== Comparación para: Busca en la base de conocimiento local qué es observabilidad.
--- Antes ---


,step,tipo,input,output,latencia,tokens,estado
0,1,decision,Busca en la base de conocimiento local qué es ...,usar herramienta local,0.05s,10,ok
1,2,tool,Busca en la base de conocimiento local qué es ...,"La observabilidad captura logs, trazas y métri...",0.10s,0,ok
2,3,llm,Busca en la base de conocimiento local qué es ...,El agente encontró datos locales y respondió c...,0.12s,35,ok


--- Después ---


,step,tipo,input,output,latencia,tokens,estado
0,1,decision,Busca en la base de conocimiento local qué es ...,usar herramienta local,0.05s,10,ok
1,2,tool,Busca en la base de conocimiento local qué es ...,"La observabilidad captura logs, trazas y métri...",0.10s,0,ok
2,3,llm,"La observabilidad captura logs, trazas y métri...",Usé la información local y respondí: La observ...,0.12s,25,ok



=== Comparación para: ¿Qué es un agente y cómo usa herramientas?
--- Antes ---


,step,tipo,input,output,latencia,tokens,estado
0,1,decision,¿Qué es un agente y cómo usa herramientas?,respuesta directa,0.05s,10,ok
1,3,llm,¿Qué es un agente y cómo usa herramientas?,El agente encontró datos locales y respondió c...,0.12s,35,ok


... (3 outputs left out)

,step,tipo,input,output,latencia,tokens,estado
0,1,decision,Consulta datos sobre historia.,respuesta directa,0.05s,10,ok
1,3,llm,Consulta datos sobre historia.,El agente encontró datos locales y respondió c...,0.12s,35,ok


--- Después ---


,step,tipo,input,output,latencia,tokens,estado
0,1,decision,Consulta datos sobre historia.,usar herramienta local,0.05s,10,ok
1,2,tool,Consulta datos sobre historia.,La independencia se celebró en 1810 en muchos ...,0.10s,0,ok
2,3,llm,La independencia se celebró en 1810 en muchos ...,Usé la información local y respondí: La indepe...,0.12s,25,ok


Failed to refresh cache entry hormold/openai-functions-agent: Connection error caused failure to GET /commits/hormold/openai-functions-agent/latest in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded with url: /commits/hormold/openai-functions-agent/latest (Caused by NameResolutionError("HTTPSConnection(host=\'api.smith.langchain.com\', port=443): Failed to resolve \'api.smith.langchain.com\' ([Errno 11001] getaddrinfo failed)"))'))
Content-Length: None
API Key: 
Failed to refresh cache entry hormold/openai-functions-agent: Connection error caused failure to GET /commits/hormold/openai-functions-agent/latest in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded with url: /commits/hormold/openai-functions-agent/latest (Caused by NameResolutionEr

## Conclusión

- **Mira la traza paso a paso.** Si la decisión dice una cosa y el paso final dice otra, hay un error.
- **Un texto puede sonar bien y estar mal.** El agente original decía que encontró datos aunque no buscó nada.
- **Los analizadores tienen límites.** Solo encuentran lo que les enseñamos a buscar.